# Experiment A (Per-Layer): ODE Trajectory Fitting Across All GPT-2 Layers

**Goal.** Extend Experiment A by fitting first-order and second-order autonomous
ODE models independently at each of GPT-2's 13 layers (embedding + 12 transformer
layers). This reveals whether certain layers admit an autonomous ODE description
and how the fit quality varies with depth.

**Hypothesis.** Early layers may show positive R² for first-order ODE (simpler,
more local dynamics). Middle layers likely show the worst R² (most context-
dependent attention). This would mirror the "bathtub" profile of the shared-
potential separator from the TMLR paper.

**Models:** Same M1–M4 as Experiment A, fitted independently per layer.

**Runtime.** ~30 min on Colab A100 (13 layers × ~2 min each).

In [ ]:
# Cell 1 — Environment setup + GDrive mount
SEED = 42

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_experiment_a_per_layer'

import os, sys, shutil, subprocess, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


_sh(f'{sys.executable} -m pip install -q transformers torch numpy '
    'scikit-learn matplotlib tqdm scipy')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth=1 -b {REPO_BRANCH} {REPO_URL} {COLAB_REPO_PATH}')
    else:
        _sh(f'cd {COLAB_REPO_PATH} && git pull --ff-only')
else:
    REPO_ROOT = Path('__file__').resolve().parents[3]
    if not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = Path.cwd()
        while not (REPO_ROOT / 'notebooks').exists() and REPO_ROOT != REPO_ROOT.parent:
            REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'dynamics_order_test' / 'results' / 'experiment_a_per_layer'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

OUT_DIR = GDRIVE_OUT / f'seed{SEED}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'SEED      = {SEED}')
print(f'Output    = {OUT_DIR}')

In [ ]:
# Cell 2 — Imports & Configuration
import math, warnings
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

CFG = dict(
    model_name      = 'gpt2',
    layers           = list(range(13)),  # 0=embedding, 1..12=transformer layers
    d_pca            = 50,
    n_train_sent     = 40,
    d_hidden_V       = 128,
    n_layers_V       = 2,
    d_hidden_gen     = 128,
    n_layers_gen     = 2,
    lr               = 1e-3,
    n_epochs         = 300,
    batch_size       = 256,
    weight_decay     = 1e-4,
    rollout_steps    = [1, 5, 10],
    seed             = SEED,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
print(f'Config saved to {OUT_DIR / "config.json"}')

In [ ]:
# Cell 3 — Corpus
CORPUS = {
  "mathematics": [
    "The fundamental theorem of calculus establishes that differentiation and integration are inverse operations of each other.",
    "A metric space is a set together with a notion of distance between its elements, usually called points, that satisfies a set of axioms.",
    "Euler's identity connects the five most important numbers in mathematics through the equation e to the power of i pi plus one equals zero.",
    "The eigenvalues of a symmetric matrix are always real, and the eigenvectors corresponding to distinct eigenvalues are orthogonal.",
    "Gödel's incompleteness theorems demonstrate that in any consistent formal system capable of expressing basic arithmetic there exist statements that can neither be proved nor disproved.",
    "The Riemann hypothesis conjectures that all non-trivial zeros of the Riemann zeta function have real part equal to one half.",
    "A group homomorphism preserves the algebraic structure by mapping the identity element to the identity element and products to products.",
    "The central limit theorem states that the sum of a large number of independent random variables tends toward a normal distribution regardless of the underlying distribution.",
    "Hilbert spaces generalize the notion of Euclidean space to infinite dimensions while retaining the structure of an inner product.",
    "The Lagrangian of a mechanical system equals the kinetic energy minus the potential energy and encodes the complete dynamics through the Euler-Lagrange equations."
  ],
  "narrative": [
    "The old lighthouse keeper climbed the spiral staircase one last time, his weathered hands gripping the iron railing as the storm gathered outside.",
    "She found the letter tucked between the pages of a book she hadn't opened in years, the ink faded but the words still sharp enough to wound.",
    "The train pulled into the empty station at midnight, its headlamp cutting through the fog like a single unblinking eye.",
    "He sat on the porch watching the fireflies trace their erratic paths through the warm summer air while the radio played something slow and sad.",
    "The market was closing for the day and the vendors were packing up their unsold fruit, bruised peaches and overripe plums going back into crates.",
    "She ran through the forest with branches whipping at her face, the sound of the river growing louder with every desperate step.",
    "The children built a fort out of couch cushions and draped a bedsheet over the top, declaring it a castle that no adults could enter.",
    "He returned to the village after twenty years and found that the oak tree in the square had been cut down and replaced by a parking lot.",
    "The ship appeared on the horizon at dawn, its sails torn and its hull battered, carrying survivors of a voyage no one had expected to end.",
    "She opened the old music box and it played the same melody her grandmother used to hum while braiding her hair on Sunday mornings."
  ],
  "scientific": [
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen using light energy absorbed by chlorophyll molecules in the thylakoid membranes.",
    "The theory of plate tectonics explains that Earth's lithosphere is divided into several large plates that float on the semi-fluid asthenosphere beneath.",
    "Antibiotics work by either killing bacteria directly or inhibiting their ability to grow and reproduce, but they have no effect on viral infections.",
    "Black holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon.",
    "The human genome contains approximately three billion base pairs of DNA organized into twenty-three pairs of chromosomes in each cell nucleus.",
    "Quantum entanglement describes a phenomenon where two particles become correlated in such a way that the quantum state of each particle cannot be described independently.",
    "Mitochondria are often called the powerhouses of the cell because they generate most of the cell's supply of adenosine triphosphate used as chemical energy.",
    "The Doppler effect explains why the pitch of an ambulance siren appears to change as the vehicle approaches and then recedes from an observer.",
    "CRISPR-Cas9 is a gene-editing technology that allows scientists to add, remove, or alter genetic material at particular locations in the genome with unprecedented precision.",
    "Dark matter makes up approximately twenty-seven percent of the universe's total mass-energy content but does not interact with electromagnetic radiation and has never been directly observed."
  ],
  "code_description": [
    "A hash table stores key-value pairs and uses a hash function to compute an index into an array of buckets from which the desired value can be found.",
    "Recursion is a technique where a function calls itself with a modified argument until it reaches a base case that returns without further recursive calls.",
    "The model-view-controller pattern separates an application into three interconnected components to separate internal representations from the ways information is presented.",
    "Gradient descent is an optimization algorithm that iteratively adjusts parameters in the direction of steepest descent of the loss function to find a local minimum.",
    "A binary search tree maintains sorted data and allows lookup, insertion, and deletion operations in time proportional to the logarithm of the number of elements.",
    "Docker containers package an application with all its dependencies into a standardized unit that can run consistently across different computing environments.",
    "Backpropagation computes the gradient of the loss function with respect to each weight by applying the chain rule layer by layer from the output back to the input.",
    "A relational database organizes data into tables of rows and columns and uses structured query language to manage and retrieve the stored information.",
    "Version control systems like Git track changes to source code over time, allowing multiple developers to collaborate on a project without overwriting each other's work.",
    "An API defines a set of rules and protocols that allows different software applications to communicate with each other by sending requests and receiving responses."
  ],
  "conversational": [
    "I think we should grab coffee sometime this week because there are a few things I've been meaning to discuss with you about the project timeline.",
    "Have you ever noticed how the same song can sound completely different depending on whether you're happy or sad when you hear it?",
    "My neighbor's dog escaped again last night and we spent two hours searching the neighborhood before finding him asleep under a parked car.",
    "The restaurant on the corner finally reopened after the renovation and honestly the food is even better now than it was before they closed.",
    "I tried to learn how to play the piano when I was a kid but gave up after about six months because I couldn't stand practicing scales.",
    "She asked me what I wanted for my birthday and I couldn't think of a single thing which made me realize I already have everything I need.",
    "The weather forecast said it would rain all weekend but the sun came out Saturday morning and stayed until Monday.",
    "I'm not sure if I should take the job offer because it pays more but the commute would add two hours to my day.",
    "We watched three movies in a row last night and by the end of the third one everyone had fallen asleep on the couch.",
    "He told me the secret to his grandmother's pasta sauce is a pinch of cinnamon which sounds strange but actually makes a big difference."
  ]
}

sentences, domains = [], []
for domain, sents in CORPUS.items():
    for s in sents:
        sentences.append(s)
        domains.append(domain)
print(f'Corpus: {len(sentences)} sentences, {len(set(domains))} domains')

In [ ]:
# Cell 4 — Extract hidden states at ALL layers from GPT-2
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
gpt2_model = AutoModelForCausalLM.from_pretrained(
    CFG['model_name'], output_hidden_states=True
).eval().to(DEVICE)

@torch.no_grad()
def extract_all_layers(sentence: str):
    """Return dict: layer_idx -> (T, d_model) numpy array."""
    toks = tokenizer(sentence, return_tensors='pt').to(DEVICE)
    out = gpt2_model(**toks)
    result = {}
    for layer_idx in CFG['layers']:
        hs = out.hidden_states[layer_idx]  # (1, T, d_model)
        result[layer_idx] = hs.squeeze(0).cpu().numpy()
    return result

# all_hidden[layer_idx][sentence_idx] = (T_i, d_model)
all_hidden = {layer: [] for layer in CFG['layers']}
for i, s in enumerate(tqdm(sentences, desc='Extracting hidden states')):
    layer_dict = extract_all_layers(s)
    for layer in CFG['layers']:
        all_hidden[layer].append(layer_dict[layer])

total_tokens = sum(h.shape[0] for h in all_hidden[0])
d_model = all_hidden[0][0].shape[1]
print(f'Extracted {total_tokens} tokens across {len(sentences)} sentences, '
      f'd_model={d_model}, {len(CFG["layers"])} layers')

del gpt2_model
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

In [ ]:
# Cell 5 — Model definitions (same as Experiment A)

class ScalarPotentialMLP(nn.Module):
    def __init__(self, d_in, d_hidden, n_layers):
        super().__init__()
        layers = []
        d = d_in
        for _ in range(n_layers):
            layers += [nn.Linear(d, d_hidden), nn.GELU()]
            d = d_hidden
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, h):
        return self.net(h).squeeze(-1)


class FirstOrderPhysics(nn.Module):
    def __init__(self, d, d_hidden, n_layers):
        super().__init__()
        self.V = ScalarPotentialMLP(d, d_hidden, n_layers)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, h_curr, h_prev=None):
        h = h_curr.detach().requires_grad_(True)
        V = self.V(h)
        grad_V = torch.autograd.grad(V.sum(), h, create_graph=True)[0]
        alpha = self.log_alpha.exp()
        return h_curr - alpha * grad_V


class SecondOrderPhysics(nn.Module):
    def __init__(self, d, d_hidden, n_layers):
        super().__init__()
        self.V = ScalarPotentialMLP(d, d_hidden, n_layers)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))
        self.log_gamma = nn.Parameter(torch.tensor(0.0))

    def forward(self, h_curr, h_prev):
        v = h_curr - h_prev
        h = h_curr.detach().requires_grad_(True)
        V = self.V(h)
        grad_V = torch.autograd.grad(V.sum(), h, create_graph=True)[0]
        alpha = self.log_alpha.exp()
        gamma = self.log_gamma.exp()
        damp = 1.0 / (1.0 + gamma)
        return h_curr + damp * v - damp * alpha * grad_V


class GeneralMLPDynamics(nn.Module):
    def __init__(self, d, d_hidden, n_layers, use_velocity=False):
        super().__init__()
        self.use_velocity = use_velocity
        d_in = 2 * d if use_velocity else d
        layers = []
        din = d_in
        for _ in range(n_layers):
            layers += [nn.Linear(din, d_hidden), nn.GELU()]
            din = d_hidden
        layers.append(nn.Linear(din, d))
        self.net = nn.Sequential(*layers)

    def forward(self, h_curr, h_prev=None):
        if self.use_velocity:
            v = h_curr - h_prev
            x = torch.cat([h_curr, v], dim=-1)
        else:
            x = h_curr
        return h_curr + self.net(x)


def make_models(d):
    return {
        'M1_1st_order_physics':  FirstOrderPhysics(d, CFG['d_hidden_V'], CFG['n_layers_V']),
        'M2_2nd_order_physics':  SecondOrderPhysics(d, CFG['d_hidden_V'], CFG['n_layers_V']),
        'M3_general_lag1':       GeneralMLPDynamics(d, CFG['d_hidden_gen'], CFG['n_layers_gen'], False),
        'M4_general_lag2':       GeneralMLPDynamics(d, CFG['d_hidden_gen'], CFG['n_layers_gen'], True),
    }

print('Model definitions loaded.')

In [ ]:
# Cell 6 — Training and evaluation functions

def compute_r2(y_true, y_pred):
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean(dim=0, keepdim=True)) ** 2).sum()
    return 1.0 - (ss_res / ss_tot).item()


def make_triplets(hidden_pca_list, indices):
    h_prev, h_curr, h_next = [], [], []
    for i in indices:
        h = hidden_pca_list[i]
        T = h.shape[0]
        if T < 3:
            continue
        for t in range(1, T - 1):
            h_prev.append(h[t-1])
            h_curr.append(h[t])
            h_next.append(h[t+1])
    return (
        torch.tensor(np.array(h_prev), dtype=torch.float32),
        torch.tensor(np.array(h_curr), dtype=torch.float32),
        torch.tensor(np.array(h_next), dtype=torch.float32),
    )


def train_and_eval_model(name, model, hp_tr, hc_tr, hn_tr, hp_te, hc_te, hn_te, cfg):
    """Train model on train triplets, return test single-step R² and MSE."""
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                            weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['n_epochs'])

    hp = hp_tr.to(DEVICE)
    hc = hc_tr.to(DEVICE)
    hn = hn_tr.to(DEVICE)
    N = hc.shape[0]
    bs = cfg['batch_size']

    for epoch in range(cfg['n_epochs']):
        model.train()
        perm = torch.randperm(N, device=DEVICE)
        for start in range(0, N, bs):
            idx = perm[start:start+bs]
            pred = model(hc[idx], hp[idx])
            loss = F.mse_loss(pred, hn[idx])
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()

    # Evaluate on test set
    model.eval()
    hp_d = hp_te.to(DEVICE)
    hc_d = hc_te.to(DEVICE)
    hn_d = hn_te.to(DEVICE)
    with torch.enable_grad():
        pred = model(hc_d, hp_d)
    mse = F.mse_loss(pred.detach(), hn_d).item()
    r2 = compute_r2(hn_d, pred.detach())

    # Extract learned params
    params = {}
    if hasattr(model, 'log_alpha'):
        params['alpha'] = model.log_alpha.exp().item()
    if hasattr(model, 'log_gamma'):
        gamma = model.log_gamma.exp().item()
        params['gamma'] = gamma
        params['velocity_retention'] = 1.0 / (1.0 + gamma)

    return {'mse': mse, 'r2': r2, 'params': params}


print('Training and evaluation functions ready.')

In [ ]:
# Cell 7 — Main per-layer sweep

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

n_train = CFG['n_train_sent']
perm = np.random.permutation(len(sentences))
train_idx = perm[:n_train]
test_idx = perm[n_train:]

model_names = ['M1_1st_order_physics', 'M2_2nd_order_physics',
               'M3_general_lag1', 'M4_general_lag2']

# Results: layer -> model_name -> {mse, r2, params}
all_results = {}
d_pca = CFG['d_pca']

t0_total = time.time()

for layer in CFG['layers']:
    t0_layer = time.time()
    print(f'\n{"="*60}')
    print(f'Layer {layer}')
    print(f'{"="*60}')

    # PCA for this layer
    layer_hidden = all_hidden[layer]  # list of (T_i, d_model)
    all_vecs = np.concatenate(layer_hidden, axis=0)
    pca = PCA(n_components=d_pca, random_state=CFG['seed'])
    pca.fit(all_vecs)
    explained = pca.explained_variance_ratio_.sum()
    hidden_pca = [pca.transform(h) for h in layer_hidden]
    print(f'  PCA-{d_pca} explains {explained:.1%} of variance')

    # Make triplets
    hp_tr, hc_tr, hn_tr = make_triplets(hidden_pca, train_idx)
    hp_te, hc_te, hn_te = make_triplets(hidden_pca, test_idx)
    print(f'  Train: {hp_tr.shape[0]} triplets, Test: {hp_te.shape[0]} triplets')

    # Train all 4 models
    layer_results = {}
    models = make_models(d_pca)
    for name in model_names:
        torch.manual_seed(CFG['seed'] + layer * 100)
        m = make_models(d_pca)[name]
        res = train_and_eval_model(name, m, hp_tr, hc_tr, hn_tr,
                                   hp_te, hc_te, hn_te, CFG)
        layer_results[name] = res
        r2_str = f'{res["r2"]:+.4f}'
        params_str = ''
        if res['params']:
            params_str = '  ' + ', '.join(f'{k}={v:.3f}' for k, v in res['params'].items())
        print(f'  {name:<25s}  R²={r2_str}  MSE={res["mse"]:.2f}{params_str}')

    all_results[layer] = {
        'pca_variance_explained': float(explained),
        'n_train_triplets': int(hp_tr.shape[0]),
        'n_test_triplets': int(hp_te.shape[0]),
        'models': {name: {k: v for k, v in r.items()}
                   for name, r in layer_results.items()},
    }

    elapsed = time.time() - t0_layer
    print(f'  Layer {layer} done in {elapsed:.1f}s')

total_elapsed = time.time() - t0_total
print(f'\nTotal sweep time: {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)')

# Save all results
save_results = {str(k): v for k, v in all_results.items()}
with open(OUT_DIR / 'per_layer_results.json', 'w') as f:
    json.dump(save_results, f, indent=2)
print(f'Saved to {OUT_DIR / "per_layer_results.json"}')

In [ ]:
# Cell 8 — Main visualization: R² layer profile for all 4 models

layers = CFG['layers']
colors = {'M1_1st_order_physics': '#1f77b4', 'M2_2nd_order_physics': '#ff7f0e',
          'M3_general_lag1': '#2ca02c', 'M4_general_lag2': '#d62728'}
labels = {'M1_1st_order_physics': 'M1: 1st-order physics',
          'M2_2nd_order_physics': 'M2: 2nd-order physics',
          'M3_general_lag1': 'M3: General lag-1 MLP',
          'M4_general_lag2': 'M4: General lag-2 MLP'}

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for name in model_names:
    r2_vals = [all_results[l]['models'][name]['r2'] for l in layers]
    ax.plot(layers, r2_vals, 'o-', color=colors[name], label=labels[name],
            linewidth=2, markersize=6)

ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('GPT-2 Layer Index (0 = embedding, 1–12 = transformer)', fontsize=12)
ax.set_ylabel('Single-step test R²', fontsize=12)
ax.set_title('Per-Layer ODE Fit Quality — First-Order vs Second-Order on GPT-2',
             fontsize=14)
ax.set_xticks(layers)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUT_DIR / 'per_layer_r2_profile.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "per_layer_r2_profile.png"}')
plt.show()

In [ ]:
# Cell 9 — Learned gamma profile (M2 only)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

gammas = [all_results[l]['models']['M2_2nd_order_physics']['params'].get('gamma', 0)
          for l in layers]
vel_ret = [all_results[l]['models']['M2_2nd_order_physics']['params'].get('velocity_retention', 0)
           for l in layers]

ax1.bar(layers, gammas, color='#ff7f0e', alpha=0.8)
ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Learned γ (damping)', fontsize=12)
ax1.set_title('Learned Damping Coefficient by Layer', fontsize=13)
ax1.set_xticks(layers)
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(layers, vel_ret, color='#ff7f0e', alpha=0.8)
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Velocity retention 1/(1+γ)', fontsize=12)
ax2.set_title('Velocity Retention by Layer', fontsize=13)
ax2.set_xticks(layers)
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% retention')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(OUT_DIR / 'per_layer_gamma_profile.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "per_layer_gamma_profile.png"}')
plt.show()

In [ ]:
# Cell 10 — PCA variance explained by layer

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
pca_vars = [all_results[l]['pca_variance_explained'] for l in layers]
ax.bar(layers, [v * 100 for v in pca_vars], color='#9467bd', alpha=0.8)
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('PCA-50 variance explained (%)', fontsize=12)
ax.set_title('PCA-50 Variance Retention by Layer', fontsize=13)
ax.set_xticks(layers)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(pca_vars):
    ax.text(i, v * 100 + 1, f'{v:.1%}', ha='center', fontsize=8)

plt.tight_layout()
fig.savefig(OUT_DIR / 'per_layer_pca_variance.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "per_layer_pca_variance.png"}')
plt.show()

In [ ]:
# Cell 11 — Summary table

print('=' * 90)
print('PER-LAYER ODE FIT SUMMARY')
print('=' * 90)
print(f'{"Layer":>5s}  {"PCA var":>8s}  ', end='')
for name in model_names:
    short = name.split('_', 1)[0]
    print(f'{short+" R²":>10s}', end='')
print(f'{"M2 γ":>8s} {"M2 vel.ret":>10s}')
print('-' * 90)

for l in layers:
    r = all_results[l]
    pca_v = r['pca_variance_explained']
    gamma = r['models']['M2_2nd_order_physics']['params'].get('gamma', 0)
    vr = r['models']['M2_2nd_order_physics']['params'].get('velocity_retention', 0)
    print(f'{l:>5d}  {pca_v:>7.1%}  ', end='')
    for name in model_names:
        r2 = r['models'][name]['r2']
        marker = ' *' if r2 > 0 else '  '
        print(f'{r2:>+8.4f}{marker}', end='')
    print(f'{gamma:>8.2f} {vr:>10.3f}')

print('-' * 90)
print('* = positive R² (autonomous ODE captures some variance at this layer)')
print()

# Count layers with positive R² per model
print('Layers with R² > 0:')
for name in model_names:
    pos_layers = [l for l in layers if all_results[l]['models'][name]['r2'] > 0]
    print(f'  {labels[name]}: {len(pos_layers)}/13  {pos_layers}')

print(f'\nAll results saved to: {OUT_DIR}')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.relative_to(OUT_DIR)}  ({size_kb:.1f} KB)')